# Library 07: report writer

We write the report text in `08_write_report.ipynb` as a list of **blocks**:

| Block | Meaning |
|---|---|
| `("h", text)` | section heading |
| `("p", text)` | paragraph |
| `("table", caption, header, rows)` | table |
| `("fig", image_path, caption)` | figure |
| `("ref", text)` | reference-list entry |

This notebook turns the blocks into three files: a **Word** file based on the course template, a **Markdown**
copy that is easy to read on GitHub, and an **HTML** file laid out for printing. Google Chrome (run in the
background, without a window) prints the HTML to the PDF we submit. All of them follow the course rules for
the report: A4, 3 cm margins, 12 pt Times, single line spacing.

Needs: `lib_00_settings`.

In [ ]:
import re
import subprocess
import urllib.parse
from html import escape as escape_html

from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Pt

REPORT_FONT = "Times New Roman"
WORD_LIMIT = 5000
CHROME_PATH = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

# A paragraph that starts with a short label such as "Validity." or "Degree and strength (RQ2)." gets that
# label in bold. The pattern only matches labels of up to three words, or labels that name a research question,
# so a normal first sentence is never turned bold by mistake.
BOLD_LABEL_PATTERN = re.compile(r"^((?:Table|Figure) \d+\.|[A-Z][A-Za-z ,\-]{2,60}?(?: \(RQ[\d, RQ]+\))?\.) ")


def split_bold_label(paragraph_text: str) -> tuple[str, str]:
    """Split a paragraph into (bold label, rest). The label is '' when the paragraph doesn't start with one."""
    match = BOLD_LABEL_PATTERN.match(paragraph_text)
    if match and (len(match.group(1).split()) <= 3 or "(RQ" in match.group(1)):
        return match.group(1), paragraph_text[match.end():]
    return "", paragraph_text

## Word file

We open the course template, clear out its example text and keep its styles (`Heading 1` for the title,
`Heading 3` for the section titles). Then we set the page to the course rules.

In [ ]:
def style_run(text_run, font_size: float, bold: bool = False, italic: bool = False) -> None:
    """Set font, size and bold/italic on one piece of text. The eastAsia font is set too, otherwise Word may
    fall back to another font for some characters."""
    text_run.font.name = REPORT_FONT
    text_run._element.rPr.rFonts.set(qn("w:eastAsia"), REPORT_FONT)
    text_run.font.size = Pt(font_size)
    text_run.bold, text_run.italic = bold, italic


def add_paragraph(document, text: str, font_size: float = 12, alignment=WD_ALIGN_PARAGRAPH.JUSTIFY,
                  space_after: float = 6, bold_label: bool = True):
    """Add one single-spaced paragraph, with its leading label in bold if it has one."""
    paragraph = document.add_paragraph(style="Normal")
    paragraph.alignment = alignment
    paragraph.paragraph_format.space_after = Pt(space_after)
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.line_spacing = 1.0
    label, rest = split_bold_label(text) if bold_label else ("", text)
    if label:
        style_run(paragraph.add_run(label + " "), font_size, bold=True)
    style_run(paragraph.add_run(rest), font_size)
    return paragraph


def add_horizontal_rules(word_table) -> None:
    """Plain table style: a line above the table, thin lines between rows and a line at the bottom."""
    table_borders = OxmlElement("w:tblBorders")
    for edge_name in ("top", "bottom", "insideH"):
        border = OxmlElement(f"w:{edge_name}")
        border.set(qn("w:val"), "single")
        border.set(qn("w:sz"), "2" if edge_name == "insideH" else "4")
        border.set(qn("w:color"), "D9D9D9" if edge_name == "insideH" else "7F7F7F")
        table_borders.append(border)
    word_table._tbl.tblPr.append(table_borders)


def has_long_cells(rows: list[list[str]]) -> bool:
    """True when a table holds sentences (like player lists) rather than numbers; those get smaller text."""
    return max(len(str(cell)) for row in rows for cell in row[1:]) > 30


def add_word_table(document, caption: str, header: list[str], rows: list[list[str]]) -> None:
    """Caption followed by the table. Numbers are right-aligned so the digits line up."""
    add_paragraph(document, caption, font_size=10, alignment=WD_ALIGN_PARAGRAPH.LEFT, space_after=3)
    word_table = document.add_table(rows=1 + len(rows), cols=len(header))
    word_table.alignment = WD_TABLE_ALIGNMENT.CENTER
    add_horizontal_rules(word_table)
    long_cells = has_long_cells(rows)
    cell_font_size = 8 if long_cells else 9.5
    for row_number, row in enumerate([header] + rows):
        for column_number, cell_text in enumerate(row):
            cell_paragraph = word_table.cell(row_number, column_number).paragraphs[0]
            cell_paragraph.paragraph_format.space_after = Pt(1)
            cell_paragraph.paragraph_format.line_spacing = 1.0
            left_aligned = column_number == 0 or long_cells
            cell_paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if left_aligned else WD_ALIGN_PARAGRAPH.RIGHT
            style_run(cell_paragraph.add_run(str(cell_text)), cell_font_size, bold=(row_number == 0))
    if long_cells:
        for table_row in word_table.rows:
            table_row.cells[0].width = Cm(2.2)
            for cell in table_row.cells[1:]:
                cell.width = Cm(4.3)
    document.add_paragraph().paragraph_format.space_after = Pt(0)


def write_word_report(blocks: list[tuple], report_details: dict, output_path: Path) -> None:
    """Build the Word version of the report from the blocks."""
    document = Document(TEMPLATE_PATH)
    document_body = document.element.body
    # Remove the template's example text. The last element (sectPr) holds the page settings, so it stays.
    for element in list(document_body):
        if element.tag != qn("w:sectPr"):
            document_body.remove(element)
    # File properties: the template still carries its own author name, so we put ours in.
    team_names = ", ".join(line.split(",")[0] for line in report_details["authors"])
    document.core_properties.author = team_names
    document.core_properties.last_modified_by = team_names
    document.core_properties.title = report_details["title"]
    page = document.sections[0]
    page.page_width, page.page_height = Cm(21.0), Cm(29.7)
    page.left_margin = page.right_margin = page.top_margin = page.bottom_margin = Cm(3)
    normal_style = document.styles["Normal"]
    normal_style.font.name, normal_style.font.size = REPORT_FONT, Pt(12)
    normal_style.element.rPr.rFonts.set(qn("w:eastAsia"), REPORT_FONT)

    title = document.add_paragraph(style="Heading 1")
    style_run(title.add_run(report_details["title"]), 16, bold=True)
    for author_line in report_details["authors"]:
        add_paragraph(document, author_line, space_after=0, alignment=WD_ALIGN_PARAGRAPH.LEFT, bold_label=False)

    for block_type, *block_content in blocks:
        if block_type == "h":
            heading = document.add_paragraph(style="Heading 3")
            style_run(heading.add_run(block_content[0]), 13, bold=True)
        elif block_type == "p":
            add_paragraph(document, block_content[0])
        elif block_type == "ref":
            add_paragraph(document, block_content[0], font_size=10, space_after=2,
                          alignment=WD_ALIGN_PARAGRAPH.LEFT, bold_label=False)
        elif block_type == "table":
            add_word_table(document, *block_content)
        elif block_type == "fig":
            picture_paragraph = document.add_paragraph()
            picture_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
            picture_paragraph.paragraph_format.space_after = Pt(2)
            picture_paragraph.add_run().add_picture(str(block_content[0]), width=Cm(15))
            add_paragraph(document, block_content[1], font_size=10, space_after=8)
    document.save(output_path)

## Markdown and HTML copies

In [ ]:
def write_markdown_report(blocks: list[tuple], report_details: dict, output_path: Path) -> None:
    """Markdown copy of the report. Image links are relative, so they work on GitHub."""
    lines = [f"# {report_details['title']}", ""] + [f"{author}  " for author in report_details["authors"]] + [""]
    for block_type, *block_content in blocks:
        if block_type == "h":
            lines += [f"## {block_content[0]}", ""]
        elif block_type in ("p", "ref"):
            lines += [block_content[0], ""]
        elif block_type == "table":
            caption, header, rows = block_content
            lines += [f"**{caption}**", "", "| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
            lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows] + [""]
        elif block_type == "fig":
            relative_image = Path(block_content[0]).relative_to(PROJECT_ROOT).as_posix()
            lines += [f"![{block_content[1][:9]}](../{relative_image})", "", f"*{block_content[1]}*", ""]
    output_path.write_text("\n".join(lines), encoding="utf-8")


PRINT_STYLES = """
@page { size: A4; margin: 3cm; }
body { font-family: 'Times New Roman', Times, serif; font-size: 12pt; line-height: 1.0; margin: 0; color: #000; }
h1 { font-size: 16pt; margin: 0 0 8pt 0; }
h3 { font-size: 13pt; margin: 12pt 0 4pt 0; break-after: avoid; }
p { margin: 0 0 6pt 0; text-align: justify; }
p.author { margin: 0; text-align: left; }
p.caption { font-size: 10pt; text-align: left; margin: 6pt 0 3pt 0; break-after: avoid; }
p.figcap { font-size: 10pt; margin: 2pt 0 8pt 0; }
p.ref { font-size: 10pt; text-align: left; margin: 0 0 2pt 0; }
table { border-collapse: collapse; width: 100%; margin: 0 0 8pt 0; font-size: 9.5pt; break-inside: avoid;
        border-top: 1px solid #7f7f7f; border-bottom: 1px solid #7f7f7f; }
table.wide { font-size: 8pt; }
th, td { padding: 1.5pt 4pt; text-align: right; vertical-align: top; border-bottom: 0.5px solid #d9d9d9; }
th:first-child, td:first-child, table.wide th, table.wide td { text-align: left; }
th { border-bottom: 1px solid #7f7f7f; }
figure { margin: 6pt 0 0 0; break-inside: avoid; }
img { width: 100%; }
"""


def html_with_bold_label(paragraph_text: str) -> str:
    """HTML for a paragraph, with its leading label in bold. All text is escaped so '<' or '&' can't break the page."""
    label, rest = split_bold_label(paragraph_text)
    return f"<b>{escape_html(label)}</b> {escape_html(rest)}" if label else escape_html(paragraph_text)


def write_html_report(blocks: list[tuple], report_details: dict, output_path: Path) -> None:
    """HTML version laid out for A4 printing. Image paths are relative to the report folder."""
    html_parts = [f"<!doctype html><html lang='en'><head><meta charset='utf-8'>"
                  f"<title>{escape_html(report_details['title'])}</title><style>{PRINT_STYLES}</style></head><body>",
                  f"<h1>{escape_html(report_details['title'])}</h1>"]
    html_parts += [f"<p class='author'>{escape_html(author)}</p>" for author in report_details["authors"]]
    for block_type, *block_content in blocks:
        if block_type == "h":
            html_parts.append(f"<h3>{escape_html(block_content[0])}</h3>")
        elif block_type == "p":
            html_parts.append(f"<p>{html_with_bold_label(block_content[0])}</p>")
        elif block_type == "ref":
            html_parts.append(f"<p class='ref'>{escape_html(block_content[0])}</p>")
        elif block_type == "table":
            caption, header, rows = block_content
            table_class = "wide" if has_long_cells(rows) else ""
            header_html = "".join(f"<th>{escape_html(cell)}</th>" for cell in header)
            rows_html = "".join("<tr>" + "".join(f"<td>{escape_html(str(cell))}</td>" for cell in row) + "</tr>"
                                for row in rows)
            html_parts.append(f"<p class='caption'>{html_with_bold_label(caption)}</p>"
                              f"<table class='{table_class}'><tr>{header_html}</tr>{rows_html}</table>")
        elif block_type == "fig":
            relative_image = Path("..") / Path(block_content[0]).relative_to(PROJECT_ROOT)
            html_parts.append(f"<figure><img src='{relative_image.as_posix()}'></figure>"
                              f"<p class='figcap'>{html_with_bold_label(block_content[1])}</p>")
    html_parts.append("</body></html>")
    output_path.write_text("\n".join(html_parts), encoding="utf-8")

## Word count and PDF

The course limit is 5,000 words. To be on the safe side we count every word that appears in the report: title,
authors, headings, paragraphs, table cells, captions and references.

In [ ]:
def count_report_words(blocks: list[tuple], report_details: dict) -> int:
    """Number of words in the whole report, tables and references included."""
    text_parts = [report_details["title"], *report_details["authors"]]
    for block_type, *block_content in blocks:
        if block_type in ("h", "p", "ref"):
            text_parts.append(block_content[0])
        elif block_type == "table":
            caption, header, rows = block_content
            text_parts += [caption, *header, *[str(cell) for row in rows for cell in row]]
        elif block_type == "fig":
            text_parts.append(block_content[1])
    return len(" ".join(text_parts).split())


def print_html_to_pdf(html_path: Path, pdf_path: Path, chrome_path: str = CHROME_PATH) -> None:
    """Print the HTML report to PDF with Google Chrome running in the background.

    The page size and margins come from the @page rule in the HTML. We switch off Chrome's own header and
    footer (date, file path), which should not appear in the submitted PDF.
    """
    file_url = "file://" + urllib.parse.quote(str(html_path))
    subprocess.run([chrome_path, "--headless", "--disable-gpu", "--no-pdf-header-footer",
                    f"--print-to-pdf={pdf_path}", file_url],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def set_pdf_details(pdf_path: Path, report_details: dict) -> None:
    """Write our title and names into the PDF's file properties.

    Chrome fills the 'Author' field with its own browser name, which looks odd in a submitted file, so we
    replace it with the group members' names.
    """
    from pypdf import PdfReader, PdfWriter

    team_names = ", ".join(line.split(",")[0] for line in report_details["authors"])
    pdf_writer = PdfWriter(clone_from=PdfReader(pdf_path))
    pdf_writer.add_metadata({"/Title": report_details["title"], "/Author": team_names,
                             "/Creator": team_names, "/Subject": "Network Analysis project report"})
    with pdf_path.open("wb") as pdf_file:
        pdf_writer.write(pdf_file)